In [16]:
import os
import platform
import torch

print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 1
    ))
else:
    print("No GPU assigned. Use Runtime > Change runtime type and select GPU.")

Python: 3.13.15
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.6


In [17]:
!git clone --depth 1 https://github.com/VOak222/responsible-ai-healthcare-llms.git

%cd /content/responsible-ai-healthcare-llms

!find data -maxdepth 4 -type f | sort

Cloning into 'responsible-ai-healthcare-llms'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/responsible-ai-healthcare-llms'
/content
find: ‘data’: No such file or directory


In [18]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece gdown

In [19]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/responsible_ai_healthcare_llms")
DATA_DIR = PROJECT_DIR / "data"
ADAPTER_DIR = PROJECT_DIR / "adapters"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [DATA_DIR, ADAPTER_DIR, CHECKPOINT_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/responsible_ai_healthcare_llms


In [20]:
!gdown "https://drive.google.com/uc?export=download&id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS" -O /content/ori_pqaa.json

!gdown "https://drive.google.com/uc?export=download&id=1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ" -O /content/ori_pqal.json

Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS
From (redirected): https://drive.google.com/uc?id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS&confirm=t&uuid=4c3cc81a-ab17-45e0-9ce6-3ed5547ffdd9
To: /content/ori_pqaa.json
 36% 193M/533M [00:01<00:02, 132MB/s]
Traceback (most recent call last):
  File "/usr/local/bin/gdown", line 8, in <module>
    sys.exit(main())
             ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/gdown/__main__.py", line 251, in main
    result = download(
        url=url,
    ...<11 lines>...
        cookies_file=cookies_file,
    )
  File "/usr/local/lib/python3.13/dist-packages/gdown/download.py", line 575, in download
    for chunk in res.iter_content(chunk_size=CHUNK_SIZE):
                 ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/requests/models.py", line 820, in generate
    yield from self.raw.stream(chunk_size

In [21]:
import json

# Download the necessary files
!gdown 15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS -O /content/ori_pqaa.json
!gdown 1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ -O /content/ori_pqal.json

with open("/content/ori_pqaa.json", encoding="utf-8") as file:
    artificial_data = json.load(file)

with open("/content/ori_pqal.json", encoding="utf-8") as file:
    labeled_data = json.load(file)

first_artificial_id = next(iter(artificial_data))
first_labeled_id = next(iter(labeled_data))

print("Artificial training rows:", len(artificial_data))
print("Labeled evaluation rows:", len(labeled_data))

print("\nArtificial fields:", artificial_data[first_artificial_id].keys())
print("Labeled fields:", labeled_data[first_labeled_id].keys())

print("\nQuestion example:")
print(artificial_data[first_artificial_id]["QUESTION"])

Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS
From (redirected): https://drive.google.com/uc?id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS&confirm=t&uuid=73f7e89f-67ad-4eca-8a4d-607bf3340762
To: /content/ori_pqaa.json
100% 533M/533M [00:06<00:00, 85.9MB/s]
Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ
From (redirected): https://drive.google.com/uc?id=1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ&confirm=t&uuid=beb566f9-a541-4189-a45e-180167125b67
To: /content/ori_pqal.json
100% 152M/152M [00:01<00:00, 146MB/s]
Artificial training rows: 211269
Labeled evaluation rows: 61249

Artificial fields: dict_keys(['QUESTION', 'CONTEXTS', 'LABELS', 'LONG_ANSWER', 'MESHES', 'final_decision'])
Labeled fields: dict_keys(['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR', 'LONG_ANSWER'])

Question example:
Are group 2 inna

In [22]:
!wget -q --show-progress \
  https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json \
  -O /content/ori_pqal_expert.json

/content/ori_pqal_e 100%[===================>]   2.46M  --.-KB/s    in 0.03s   


In [23]:
with open("/content/ori_pqal_expert.json", encoding="utf-8") as file:
    expert_data = json.load(file)

first_expert_id = next(iter(expert_data))

print("Expert-labeled evaluation rows:", len(expert_data))
print("Expert fields:", expert_data[first_expert_id].keys())

print("\nExpert question example:")
print(expert_data[first_expert_id]["QUESTION"])

print("\nExpert final decision:")
print(expert_data[first_expert_id]["final_decision"])

Expert-labeled evaluation rows: 1000
Expert fields: dict_keys(['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR', 'reasoning_required_pred', 'reasoning_free_pred', 'final_decision', 'LONG_ANSWER'])

Expert question example:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Expert final decision:
yes


In [24]:
from collections import Counter, defaultdict
from datasets import Dataset
from pathlib import Path
import random

SEED = 42
TRAIN_SIZE = 500
VALIDATION_SIZE = 100
CONTEXT_LIMIT_CHARS = 6000
ANSWER_LIMIT_CHARS = 1600

PREPARED_DIR = DATA_DIR / "prepared_pubmedqa"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)


def add_source_ids(data_dict):
    return [
        {"source_id": str(source_id), **row}
        for source_id, row in data_dict.items()
        if str(row.get("final_decision", "")).lower() in {"yes", "no", "maybe"}
    ]


def stratified_sample(rows, size, seed):
    rng = random.Random(seed)
    groups = defaultdict(list)

    for row in rows:
        groups[row["final_decision"].lower()].append(row)

    labels = sorted(groups)
    for label in labels:
        rng.shuffle(groups[label])

    selected = []
    base_size = size // len(labels)
    remainder = size % len(labels)

    for position, label in enumerate(labels):
        take = base_size + (1 if position < remainder else 0)
        selected.extend(groups[label][:take])

    rng.shuffle(selected)
    return selected


def format_for_sft(row):
    evidence = "\n".join(row["CONTEXTS"])[:CONTEXT_LIMIT_CHARS]
    decision = row["final_decision"].lower()
    rationale = row["LONG_ANSWER"][:ANSWER_LIMIT_CHARS]

    return {
        "source_id": row["source_id"],
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a biomedical research assistant. Use only the supplied "
                    "PubMed abstract evidence. Give a yes, no, or maybe decision and "
                    "a concise evidence-grounded rationale. Do not invent facts."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Research question:\n{row['QUESTION']}\n\n"
                    f"PubMed abstract evidence:\n{evidence}"
                ),
            },
            {
                "role": "assistant",
                "content": f"Decision: {decision}\nRationale: {rationale}",
            },
        ],
    }


artificial_rows = add_source_ids(artificial_data)
expert_rows = add_source_ids(expert_data)

train_rows = stratified_sample(artificial_rows, TRAIN_SIZE, SEED)
validation_rows = stratified_sample(expert_rows, VALIDATION_SIZE, SEED)

validation_ids = {row["source_id"] for row in validation_rows}
test_rows = [row for row in expert_rows if row["source_id"] not in validation_ids]

train_dataset = Dataset.from_list([format_for_sft(row) for row in train_rows])
validation_dataset = Dataset.from_list([format_for_sft(row) for row in validation_rows])
test_dataset = Dataset.from_list([format_for_sft(row) for row in test_rows])

train_dataset.to_json(str(PREPARED_DIR / "pubmedqa_train_500.jsonl"))
validation_dataset.to_json(str(PREPARED_DIR / "pubmedqa_validation_100.jsonl"))
test_dataset.to_json(str(PREPARED_DIR / "pubmedqa_expert_test_900.jsonl"))

print("Training rows:", len(train_dataset))
print("Validation rows:", len(validation_dataset))
print("Untouched expert test rows:", len(test_dataset))

print("\nTraining label balance:")
print(Counter(row["final_decision"].lower() for row in train_rows))

print("\nExample training record:")
print(train_dataset[0]["messages"][1]["content"][:500])
print("\nTarget:")
print(train_dataset[0]["messages"][2]["content"][:400])

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Training rows: 500
Validation rows: 100
Untouched expert test rows: 900

Training label balance:
Counter({'yes': 250, 'no': 250})

Example training record:
Research question:
Is mass-like peripheral zone enhancement on CT predictive of higher-grade ( Gleason 4 + 3 and higher ) prostate cancer?

PubMed abstract evidence:
To determine whether focal peripheral zone enhancement on routine venous-phase CT is predictive of higher-grade (Gleason 4 + 3 and higher) prostate cancer.
IRB approval was obtained and informed consent waived for this HIPAA-compliant retrospective study. Forty-three patients with higher-grade prostate cancer (≥Gleason 4 + 3) and 96

Target:
Decision: yes
Rationale: Focal mass-like peripheral zone enhancement on routine venous-phase CT is specific and predictive of higher-grade (Gleason 4 + 3 and higher) prostate cancer.


In [25]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SMOKE_RUN_NAME = "qwen2_5_1_5b_pubmedqa_lora_smoke_500"

torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

model.config.use_cache = False
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("Loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Allocated GPU memory (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct
GPU: Tesla T4
Allocated GPU memory (GB): 1.07


In [28]:
from trl import SFTConfig, SFTTrainer

RUN_DIR = CHECKPOINT_DIR / SMOKE_RUN_NAME
ADAPTER_OUTPUT_DIR = ADAPTER_DIR / SMOKE_RUN_NAME

training_args = SFTConfig(
    output_dir=str(RUN_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=4,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    max_length=1024,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    fp16=True,
    assistant_only_loss=True,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()

train_result = trainer.train()

trainer.save_model(str(ADAPTER_OUTPUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUTPUT_DIR))

print("\nSmoke-test training complete.")
print("Adapter saved to:", ADAPTER_OUTPUT_DIR)
print("Final training metrics:", train_result.metrics)

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [27]:
import inspect
import trl
import transformers
import peft
from trl import SFTConfig, SFTTrainer

print("TRL version:", trl.__version__)
print("Transformers version:", transformers.__version__)
print("PEFT version:", peft.__version__)

print("\nSFTConfig parameters:")
print(inspect.signature(SFTConfig))

TRL version: 1.13.0
Transformers version: 5.17.0
PEFT version: 0.20.0

SFTConfig parameters:
(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpoint

In [29]:
from collections import Counter
import torch

print("Trainable parameter dtypes before correction:")
print(Counter(
    str(parameter.dtype)
    for parameter in trainer.model.parameters()
    if parameter.requires_grad
))

trainer.args.fp16 = True
trainer.args.bf16 = False

for parameter in trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

print("\nTrainable parameter dtypes after correction:")
print(Counter(
    str(parameter.dtype)
    for parameter in trainer.model.parameters()
    if parameter.requires_grad
))

torch.cuda.empty_cache()

train_result = trainer.train()

trainer.save_model(str(ADAPTER_OUTPUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUTPUT_DIR))

print("\nSmoke-test training complete.")
print("Adapter saved to:", ADAPTER_OUTPUT_DIR)
print("Final training metrics:", train_result.metrics)

Trainable parameter dtypes before correction:
Counter({'torch.bfloat16': 392})

Trainable parameter dtypes after correction:
Counter({'torch.float32': 392})


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,1.367902,1.626957,1.515722,98444.000000,0.632103
50,1.544276,1.616384,1.546191,192782.000000,0.634485
63,1.442470,1.615467,1.543234,236643.000000,0.634399



Smoke-test training complete.
Adapter saved to: /content/drive/MyDrive/responsible_ai_healthcare_llms/adapters/qwen2_5_1_5b_pubmedqa_lora_smoke_500
Final training metrics: {'train_runtime': 434.0987, 'train_samples_per_second': 1.15, 'train_steps_per_second': 0.145, 'total_flos': 1886714943556608.0, 'train_loss': 1.460718412247915, 'epoch': 1.0}


In [30]:
import re
import pandas as pd
import torch


def extract_decision(text):
    cleaned = str(text).lower().strip()

    match = re.search(r"decision\s*:\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    match = re.search(r"^\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    return "unparsed"


def extract_target(messages):
    return extract_decision(messages[2]["content"])


def evaluate_pubmedqa_model(model_to_test, dataset, model_name):
    records = []
    model_to_test.eval()
    model_to_test.config.use_cache = True

    for index, row in enumerate(dataset, start=1):
        messages = [
            dict(row["messages"][0]),
            dict(row["messages"][1]),
        ]

        messages[1]["content"] += (
            "\n\nReturn exactly this format:\n"
            "Decision: yes, no, or maybe\n"
            "Rationale: one concise sentence"
        )

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model_to_test.device)

        with torch.inference_mode():
            generated_ids = model_to_test.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]
        response = tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()

        records.append({
            "row_number": index,
            "source_id": row["source_id"],
            "true_decision": extract_target(row["messages"]),
            "predicted_decision": extract_decision(response),
            "raw_response": response,
        })

        if index % 10 == 0:
            print(f"{model_name}: completed {index}/{len(dataset)} rows")

    result = pd.DataFrame(records)
    parsed = result[result["predicted_decision"] != "unparsed"].copy()

    metrics = {
        "model": model_name,
        "rows": len(result),
        "parsed_rows": len(parsed),
        "parse_success": round(len(parsed) / len(result), 3),
        "decision_accuracy": round(
            (parsed["true_decision"] == parsed["predicted_decision"]).mean(), 3
        ) if len(parsed) else 0.0,
    }

    return result, metrics

In [31]:
adapter_model = trainer.model

adapter_validation_results, adapter_metrics = evaluate_pubmedqa_model(
    adapter_model,
    validation_dataset,
    "Qwen 1.5B + PubMedQA LoRA",
)

adapter_results_path = REPORT_DIR / "qwen_1_5b_pubmedqa_lora_smoke_500_validation_predictions.csv"
adapter_validation_results.to_csv(adapter_results_path, index=False)

print(adapter_metrics)
print("Saved:", adapter_results_path)

Qwen 1.5B + PubMedQA LoRA: completed 10/100 rows
Qwen 1.5B + PubMedQA LoRA: completed 20/100 rows
Qwen 1.5B + PubMedQA LoRA: completed 30/100 rows
Qwen 1.5B + PubMedQA LoRA: completed 40/100 rows
Qwen 1.5B + PubMedQA LoRA: completed 50/100 rows
Qwen 1.5B + PubMedQA LoRA: completed 60/100 rows
Qwen 1.5B + PubMedQA LoRA: completed 70/100 rows
Qwen 1.5B + PubMedQA LoRA: completed 80/100 rows
Qwen 1.5B + PubMedQA LoRA: completed 90/100 rows
Qwen 1.5B + PubMedQA LoRA: completed 100/100 rows
{'model': 'Qwen 1.5B + PubMedQA LoRA', 'rows': 100, 'parsed_rows': 100, 'parse_success': 1.0, 'decision_accuracy': np.float64(0.36)}
Saved: /content/drive/MyDrive/responsible_ai_healthcare_llms/reports/qwen_1_5b_pubmedqa_lora_smoke_500_validation_predictions.csv


In [32]:
del adapter_model
del trainer
del model
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

base_validation_results, base_metrics = evaluate_pubmedqa_model(
    base_model,
    validation_dataset,
    "Base Qwen 1.5B",
)

base_results_path = REPORT_DIR / "qwen_1_5b_base_validation_predictions.csv"
base_validation_results.to_csv(base_results_path, index=False)

comparison = pd.DataFrame([base_metrics, adapter_metrics])
comparison_path = REPORT_DIR / "qwen_1_5b_pubmedqa_lora_smoke_500_comparison.csv"
comparison.to_csv(comparison_path, index=False)

print(comparison.to_string(index=False))
print("\nSaved:", base_results_path)
print("Saved:", comparison_path)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base Qwen 1.5B: completed 10/100 rows
Base Qwen 1.5B: completed 20/100 rows
Base Qwen 1.5B: completed 30/100 rows
Base Qwen 1.5B: completed 40/100 rows
Base Qwen 1.5B: completed 50/100 rows
Base Qwen 1.5B: completed 60/100 rows
Base Qwen 1.5B: completed 70/100 rows
Base Qwen 1.5B: completed 80/100 rows
Base Qwen 1.5B: completed 90/100 rows
Base Qwen 1.5B: completed 100/100 rows
                    model  rows  parsed_rows  parse_success  decision_accuracy
           Base Qwen 1.5B   100          100            1.0               0.36
Qwen 1.5B + PubMedQA LoRA   100          100            1.0               0.36

Saved: /content/drive/MyDrive/responsible_ai_healthcare_llms/reports/qwen_1_5b_base_validation_predictions.csv
Saved: /content/drive/MyDrive/responsible_ai_healthcare_llms/reports/qwen_1_5b_pubmedqa_lora_smoke_500_comparison.csv


In [33]:
import pandas as pd

base_df = pd.read_csv(base_results_path)
adapter_df = pd.read_csv(adapter_results_path)

comparison_detail = base_df.merge(
    adapter_df[
        ["source_id", "predicted_decision", "raw_response"]
    ],
    on="source_id",
    suffixes=("_base", "_lora"),
)

comparison_detail["base_correct"] = (
    comparison_detail["true_decision"]
    == comparison_detail["predicted_decision_base"]
)

comparison_detail["lora_correct"] = (
    comparison_detail["true_decision"]
    == comparison_detail["predicted_decision_lora"]
)

comparison_detail["same_prediction"] = (
    comparison_detail["predicted_decision_base"]
    == comparison_detail["predicted_decision_lora"]
)

comparison_detail["comparison_outcome"] = "same_result"
comparison_detail.loc[
    (~comparison_detail["base_correct"]) & comparison_detail["lora_correct"],
    "comparison_outcome",
] = "lora_improved"

comparison_detail.loc[
    comparison_detail["base_correct"] & (~comparison_detail["lora_correct"]),
    "comparison_outcome",
] = "lora_regressed"

print("Base prediction distribution:")
print(comparison_detail["predicted_decision_base"].value_counts())

print("\nLoRA prediction distribution:")
print(comparison_detail["predicted_decision_lora"].value_counts())

print("\nPrediction agreement:")
print(comparison_detail["same_prediction"].value_counts())

print("\nComparison outcome:")
print(comparison_detail["comparison_outcome"].value_counts())

print("\nLoRA confusion matrix:")
print(pd.crosstab(
    comparison_detail["true_decision"],
    comparison_detail["predicted_decision_lora"],
    rownames=["True"],
    colnames=["Predicted"],
))

detail_path = REPORT_DIR / "qwen_1_5b_pubmedqa_lora_smoke_500_error_analysis.csv"
comparison_detail.to_csv(detail_path, index=False)

print("\nSaved:", detail_path)

Base prediction distribution:
predicted_decision_base
yes    93
no      7
Name: count, dtype: int64

LoRA prediction distribution:
predicted_decision_lora
yes    96
no      4
Name: count, dtype: int64

Prediction agreement:
same_prediction
True     91
False     9
Name: count, dtype: int64

Comparison outcome:
comparison_outcome
same_result       94
lora_improved      3
lora_regressed     3
Name: count, dtype: int64

LoRA confusion matrix:
Predicted  no  yes
True              
maybe       1   33
no          3   30
yes         0   33

Saved: /content/drive/MyDrive/responsible_ai_healthcare_llms/reports/qwen_1_5b_pubmedqa_lora_smoke_500_error_analysis.csv


In [34]:
EXPERT_TRAIN_SIZE = 600
EXPERT_VALIDATION_SIZE = 200
ARTIFICIAL_STAGE1_SIZE = 5000

expert_train_rows = stratified_sample(
    expert_rows,
    EXPERT_TRAIN_SIZE,
    SEED,
)

expert_train_ids = {row["source_id"] for row in expert_train_rows}
expert_remaining_rows = [
    row for row in expert_rows
    if row["source_id"] not in expert_train_ids
]

expert_validation_rows = stratified_sample(
    expert_remaining_rows,
    EXPERT_VALIDATION_SIZE,
    SEED + 1,
)

expert_validation_ids = {
    row["source_id"] for row in expert_validation_rows
}

expert_test_rows = [
    row for row in expert_remaining_rows
    if row["source_id"] not in expert_validation_ids
]

artificial_stage1_rows = stratified_sample(
    artificial_rows,
    ARTIFICIAL_STAGE1_SIZE,
    SEED,
)

artificial_stage1_dataset = Dataset.from_list([
    format_for_sft(row) for row in artificial_stage1_rows
])

expert_train_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_train_rows
])

expert_validation_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_validation_rows
])

expert_test_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_test_rows
])

artificial_stage1_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_artificial_stage1_5000.jsonl")
)

expert_train_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_stage2_train_600.jsonl")
)

expert_validation_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_validation_200.jsonl")
)

expert_test_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_test_200.jsonl")
)

print("Stage 1 artificial training rows:", len(artificial_stage1_dataset))
print("Stage 2 expert training rows:", len(expert_train_dataset))
print("Expert validation rows:", len(expert_validation_dataset))
print("Untouched expert test rows:", len(expert_test_dataset))

print("\nStage 1 label balance:")
print(Counter(row["final_decision"].lower() for row in artificial_stage1_rows))

print("\nStage 2 expert label balance:")
print(Counter(row["final_decision"].lower() for row in expert_train_rows))

print("\nExpert validation label balance:")
print(Counter(row["final_decision"].lower() for row in expert_validation_rows))

print("\nExpert test label balance:")
print(Counter(row["final_decision"].lower() for row in expert_test_rows))

Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Stage 1 artificial training rows: 5000
Stage 2 expert training rows: 510
Expert validation rows: 200
Untouched expert test rows: 290

Stage 1 label balance:
Counter({'no': 2500, 'yes': 2500})

Stage 2 expert label balance:
Counter({'yes': 200, 'no': 200, 'maybe': 110})

Expert validation label balance:
Counter({'yes': 100, 'no': 100})

Expert test label balance:
Counter({'yes': 252, 'no': 38})


In [35]:
from sklearn.model_selection import train_test_split

expert_labels = [
    row["final_decision"].lower()
    for row in expert_rows
]

expert_train_rows, expert_remaining_rows = train_test_split(
    expert_rows,
    train_size=0.60,
    random_state=SEED,
    stratify=expert_labels,
)

remaining_labels = [
    row["final_decision"].lower()
    for row in expert_remaining_rows
]

expert_validation_rows, expert_test_rows = train_test_split(
    expert_remaining_rows,
    test_size=0.50,
    random_state=SEED + 1,
    stratify=remaining_labels,
)

expert_train_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_train_rows
])

expert_validation_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_validation_rows
])

expert_test_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_test_rows
])

expert_train_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_stage2_train_600.jsonl")
)

expert_validation_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_validation_200.jsonl")
)

expert_test_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_test_200.jsonl")
)

for split_name, split_rows in [
    ("Expert train", expert_train_rows),
    ("Expert validation", expert_validation_rows),
    ("Expert test", expert_test_rows),
]:
    label_counts = Counter(
        row["final_decision"].lower()
        for row in split_rows
    )

    print(f"{split_name}: {len(split_rows)} rows")
    print(label_counts)

    assert {"yes", "no", "maybe"}.issubset(label_counts), (
        f"{split_name} is missing a decision class."
    )

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Expert train: 600 rows
Counter({'yes': 331, 'no': 203, 'maybe': 66})
Expert validation: 200 rows
Counter({'yes': 111, 'no': 67, 'maybe': 22})
Expert test: 200 rows
Counter({'yes': 110, 'no': 68, 'maybe': 22})


In [36]:
import gc
import torch
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

try:
    del base_model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

STAGE1_NAME = "qwen2_5_1_5b_pubmedqa_stage1_artificial_5000"
STAGE1_RUN_DIR = CHECKPOINT_DIR / STAGE1_NAME
STAGE1_ADAPTER_DIR = ADAPTER_DIR / STAGE1_NAME

stage1_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

stage1_model.config.use_cache = False
stage1_model.enable_input_require_grads()

stage1_args = SFTConfig(
    output_dir=str(STAGE1_RUN_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    max_length=1024,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    assistant_only_loss=True,
    report_to="none",
    seed=SEED,
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    args=stage1_args,
    train_dataset=artificial_stage1_dataset,
    eval_dataset=expert_validation_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

# Keep the trainable LoRA adapters in stable precision on the T4 GPU.
for parameter in stage1_trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

stage1_trainer.model.print_trainable_parameters()

stage1_result = stage1_trainer.train()

stage1_trainer.save_model(str(STAGE1_ADAPTER_DIR))
tokenizer.save_pretrained(str(STAGE1_ADAPTER_DIR))

print("\nStage 1 complete.")
print("Adapter saved to:", STAGE1_ADAPTER_DIR)
print("Training metrics:", stage1_result.metrics)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.406545,1.558309,1.544544,736355.000000,0.644797
400,1.441350,1.553068,1.527016,1486592.000000,0.644792
600,1.433570,1.550363,1.533912,2239422.000000,0.644721
625,1.412349,1.550365,1.534036,2333335.000000,0.645235



Stage 1 complete.
Adapter saved to: /content/drive/MyDrive/responsible_ai_healthcare_llms/adapters/qwen2_5_1_5b_pubmedqa_stage1_artificial_5000
Training metrics: {'train_runtime': 3775.8296, 'train_samples_per_second': 1.324, 'train_steps_per_second': 0.166, 'total_flos': 1.860328855205376e+16, 'train_loss': 1.4257245727539063, 'epoch': 1.0}


In [37]:
import gc
import torch
from collections import Counter
from datasets import concatenate_datasets
from transformers import AutoModelForCausalLM
from peft import PeftModel
from trl import SFTConfig, SFTTrainer

# Free the Stage 1 model from GPU memory before loading it again.
try:
    del stage1_trainer
    del stage1_model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

# -------------------------------------------------------------------
# Balance the expert training data.
# "Maybe" is the smallest but clinically important class, so duplicate
# it twice. This makes it less likely the model will answer "yes" to
# almost everything, as it did in the smoke test.
# -------------------------------------------------------------------
maybe_train_dataset = expert_train_dataset.filter(
    lambda row: "decision: maybe" in row["messages"][-1]["content"].lower()
)

stage2_train_dataset = concatenate_datasets([
    expert_train_dataset,
    maybe_train_dataset,
    maybe_train_dataset,
]).shuffle(seed=SEED)

def get_decision(row):
    first_line = row["messages"][-1]["content"].splitlines()[0]
    return first_line.replace("Decision:", "").strip().lower()

print("Stage 2 training rows:", len(stage2_train_dataset))
print("Stage 2 label balance:", Counter(get_decision(row) for row in stage2_train_dataset))

# -------------------------------------------------------------------
# Load the Stage 1 LoRA adapter as trainable, then continue training it.
# -------------------------------------------------------------------
STAGE2_NAME = "qwen2_5_1_5b_pubmedqa_stage2_expert_calibrated"
STAGE2_RUN_DIR = CHECKPOINT_DIR / STAGE2_NAME
STAGE2_ADAPTER_DIR = ADAPTER_DIR / STAGE2_NAME

stage2_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

stage2_model = PeftModel.from_pretrained(
    stage2_base_model,
    str(STAGE1_ADAPTER_DIR),
    is_trainable=True,
)

stage2_model.config.use_cache = False
stage2_model.enable_input_require_grads()

stage2_args = SFTConfig(
    output_dir=str(STAGE2_RUN_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=75,
    save_strategy="steps",
    save_steps=75,
    save_total_limit=2,
    max_length=1024,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    assistant_only_loss=True,
    report_to="none",
    seed=SEED,
)

stage2_trainer = SFTTrainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=stage2_train_dataset,
    eval_dataset=expert_validation_dataset,
    processing_class=tokenizer,
)

# Required for stable mixed-precision training on the Colab T4.
for parameter in stage2_trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

stage2_trainer.model.print_trainable_parameters()

stage2_result = stage2_trainer.train()

stage2_trainer.save_model(str(STAGE2_ADAPTER_DIR))
tokenizer.save_pretrained(str(STAGE2_ADAPTER_DIR))

print("\nStage 2 complete.")
print("Adapter saved to:", STAGE2_ADAPTER_DIR)
print("Training metrics:", stage2_result.metrics)

Filter:   0%|          | 0/600 [00:00<?, ? examples/s]

Stage 2 training rows: 732
Stage 2 label balance: Counter({'yes': 331, 'no': 203, 'maybe': 198})


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/732 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/732 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
75,1.611966,1.542245,1.541170,266673.000000,0.642014
150,1.452115,1.547702,1.480780,533075.000000,0.642633
225,1.413822,1.555614,1.438170,797747.000000,0.641163
276,1.371180,1.555434,1.426803,979386.000000,0.641492



Stage 2 complete.
Adapter saved to: /content/drive/MyDrive/responsible_ai_healthcare_llms/adapters/qwen2_5_1_5b_pubmedqa_stage2_expert_calibrated
Training metrics: {'train_runtime': 1756.2848, 'train_samples_per_second': 1.25, 'train_steps_per_second': 0.157, 'total_flos': 7808480291874816.0, 'train_loss': 1.4928086745566216, 'epoch': 3.0}


In [38]:
import gc
import re
import torch
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from transformers import AutoModelForCausalLM
from peft import PeftModel

# ---------------------------------------------------------------
# Settings
# ---------------------------------------------------------------
FINAL_ADAPTER_DIR = ADAPTER_DIR / "qwen2_5_1_5b_pubmedqa_stage2_expert_calibrated"

PREDICTIONS_PATH = REPORT_DIR / "qwen2_5_1_5b_pubmedqa_expert_test_200_predictions.csv"
SUMMARY_PATH = REPORT_DIR / "qwen2_5_1_5b_pubmedqa_expert_test_200_summary.csv"

CLASS_LABELS = ["yes", "no", "maybe"]

# ---------------------------------------------------------------
# Extract the true label from the hidden assistant target.
# We will send only the system + user messages to the models.
# ---------------------------------------------------------------
def extract_true_decision(row):
    assistant_text = row["messages"][-1]["content"].lower()
    match = re.search(r"decision\s*:\s*(yes|no|maybe)", assistant_text)
    if not match:
        raise ValueError(f"Could not find true decision in: {assistant_text[:150]}")
    return match.group(1)

def parse_predicted_decision(text):
    cleaned = text.strip().lower()

    # Prefer an explicit "Decision: yes/no/maybe" response.
    match = re.search(r"decision\s*:\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    # Otherwise accept a direct first-word answer.
    match = re.search(r"^\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    return "unparsed"

@torch.inference_mode()
def evaluate_model(model, model_name, test_dataset):
    model.eval()
    rows = []

    for index, row in enumerate(test_dataset, start=1):
        prompt_messages = row["messages"][:2]  # system + user only

        prompt = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        )

        device = next(model.parameters()).device
        inputs = {key: value.to(device) for key, value in inputs.items()}

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

        new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]
        raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        true_decision = extract_true_decision(row)
        predicted_decision = parse_predicted_decision(raw_response)

        rows.append({
            "model": model_name,
            "row_number": index,
            "true_decision": true_decision,
            "predicted_decision": predicted_decision,
            "is_correct": predicted_decision == true_decision,
            "raw_response": raw_response,
        })

        if index % 20 == 0:
            print(f"{model_name}: completed {index}/{len(test_dataset)} rows")

    return pd.DataFrame(rows)

def summarize_results(results_df):
    y_true = results_df["true_decision"]
    y_pred = results_df["predicted_decision"]

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        zero_division=0,
    )

    summary = {
        "model": results_df["model"].iloc[0],
        "rows": len(results_df),
        "parse_success": round((y_pred != "unparsed").mean(), 3),
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "macro_precision": round(precision.mean(), 3),
        "macro_recall": round(recall.mean(), 3),
        "macro_f1": round(f1.mean(), 3),
    }

    for label, label_precision, label_recall, label_f1, label_support in zip(
        CLASS_LABELS, precision, recall, f1, support
    ):
        summary[f"{label}_precision"] = round(label_precision, 3)
        summary[f"{label}_recall"] = round(label_recall, 3)
        summary[f"{label}_f1"] = round(label_f1, 3)
        summary[f"{label}_support"] = int(label_support)

    return summary

# ---------------------------------------------------------------
# 1. Evaluate untouched base Qwen model.
# ---------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

base_results = evaluate_model(
    base_model,
    "Base Qwen2.5-1.5B",
    expert_test_dataset,
)

del base_model
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------------
# 2. Evaluate the final Stage 2 LoRA adapter.
# ---------------------------------------------------------------
adapter_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

adapter_model = PeftModel.from_pretrained(
    adapter_base_model,
    str(FINAL_ADAPTER_DIR),
)

adapter_results = evaluate_model(
    adapter_model,
    "Qwen2.5-1.5B + PubMedQA LoRA",
    expert_test_dataset,
)

del adapter_model
del adapter_base_model
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------------
# 3. Save and display the fair comparison.
# ---------------------------------------------------------------
all_results = pd.concat([base_results, adapter_results], ignore_index=True)

summary_df = pd.DataFrame([
    summarize_results(base_results),
    summarize_results(adapter_results),
])

all_results.to_csv(PREDICTIONS_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\nFINAL UNTOUCHED EXPERT-TEST COMPARISON")
print(summary_df.to_string(index=False))

for model_name, model_results in all_results.groupby("model"):
    matrix = confusion_matrix(
        model_results["true_decision"],
        model_results["predicted_decision"],
        labels=CLASS_LABELS + ["unparsed"],
    )

    print(f"\n{model_name} confusion matrix")
    print(
        pd.DataFrame(
            matrix,
            index=[f"true_{label}" for label in CLASS_LABELS + ["unparsed"]],
            columns=[f"pred_{label}" for label in CLASS_LABELS + ["unparsed"]],
        )
    )

print("\nSaved predictions:", PREDICTIONS_PATH)
print("Saved summary:", SUMMARY_PATH)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base Qwen2.5-1.5B: completed 20/200 rows
Base Qwen2.5-1.5B: completed 40/200 rows
Base Qwen2.5-1.5B: completed 60/200 rows
Base Qwen2.5-1.5B: completed 80/200 rows
Base Qwen2.5-1.5B: completed 100/200 rows
Base Qwen2.5-1.5B: completed 120/200 rows
Base Qwen2.5-1.5B: completed 140/200 rows
Base Qwen2.5-1.5B: completed 160/200 rows
Base Qwen2.5-1.5B: completed 180/200 rows
Base Qwen2.5-1.5B: completed 200/200 rows


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2.5-1.5B + PubMedQA LoRA: completed 20/200 rows
Qwen2.5-1.5B + PubMedQA LoRA: completed 40/200 rows
Qwen2.5-1.5B + PubMedQA LoRA: completed 60/200 rows
Qwen2.5-1.5B + PubMedQA LoRA: completed 80/200 rows
Qwen2.5-1.5B + PubMedQA LoRA: completed 100/200 rows
Qwen2.5-1.5B + PubMedQA LoRA: completed 120/200 rows
Qwen2.5-1.5B + PubMedQA LoRA: completed 140/200 rows
Qwen2.5-1.5B + PubMedQA LoRA: completed 160/200 rows
Qwen2.5-1.5B + PubMedQA LoRA: completed 180/200 rows
Qwen2.5-1.5B + PubMedQA LoRA: completed 200/200 rows

FINAL UNTOUCHED EXPERT-TEST COMPARISON
                       model  rows  parse_success  accuracy  macro_precision  macro_recall  macro_f1  yes_precision  yes_recall  yes_f1  yes_support  no_precision  no_recall  no_f1  no_support  maybe_precision  maybe_recall  maybe_f1  maybe_support
           Base Qwen2.5-1.5B   200            1.0     0.555            0.352         0.338     0.248          0.556       1.000   0.714          110         0.500      0.015  0.029     

In [39]:
from pathlib import Path

FINAL_REPORT_PATH = REPORT_DIR / "qwen2_5_1_5b_pubmedqa_lora_training_report.md"

report_text = """# Qwen2.5-1.5B PubMedQA LoRA Fine-Tuning Report

## Objective
Fine-tune Qwen2.5-1.5B-Instruct on PubMedQA using QLoRA/LoRA and compare it fairly with the unchanged base model.

## Compute Environment
- Platform: Google Colab
- GPU: Tesla T4, 14.6 GB VRAM
- Fine-tuning method: QLoRA with LoRA adapters
- Trainable parameters: 18,464,768 (1.182% of total parameters)

## Training Protocol

### Stage 1: Biomedical adaptation
- Dataset: 5,000 PubMedQA artificial examples
- Epochs: 1
- Runtime: 3,775.83 seconds (62.9 minutes)
- Final training loss: 1.4257

### Stage 2: Expert calibration
- Dataset: PubMedQA expert-labelled data
- Labels: yes, no, maybe
- Training rows: 732 after transparent oversampling of the smaller maybe class
- Epochs: 3
- Runtime: 1,756.28 seconds (29.3 minutes)
- Final training loss: 1.4928
- Best observed validation loss: 1.5422 at step 75

## Untouched 200-Row Expert Test Results

| Model | Accuracy | Macro-F1 | Yes Recall | No Recall | Maybe Recall |
|---|---:|---:|---:|---:|---:|
| Base Qwen2.5-1.5B | 0.555 | 0.248 | 1.000 | 0.015 | 0.000 |
| Qwen2.5-1.5B + PubMedQA LoRA | 0.735 | 0.602 | 0.845 | 0.721 | 0.227 |

## Interpretation
The LoRA adapter improved accuracy by 18 percentage points and macro-F1 by 0.354 on the same untouched expert test set. The base model was heavily biased toward predicting yes. Fine-tuning substantially improved recognition of no decisions and began recognising maybe decisions. The maybe class remains the weakest class and should be monitored in future experiments.

## Next Steps
1. Run the same PubMedQA LoRA protocol for Qwen2.5-3B-Instruct.
2. Rerun Med-HALT using the fine-tuned adapter to check for safety regression or improvement.
3. Evaluate fairness using EquityMedQA paired examples.
"""

FINAL_REPORT_PATH.write_text(report_text, encoding="utf-8")
print("Saved:", FINAL_REPORT_PATH)

Saved: /content/drive/MyDrive/responsible_ai_healthcare_llms/reports/qwen2_5_1_5b_pubmedqa_lora_training_report.md
